In [0]:
%pip install --upgrade "mlflow[databricks]>=3.1.0"
%restart_python

In [0]:
import mlflow
import mlflow.genai.datasets
from pyspark.sql import functions as F
from databricks.connect import DatabricksSession
import pandas as pd
import math

catalog = "users"
schema = "alex_miller"
table = "wikipedia_source"
evaluation_table = "dspy_rag_eval_example"

# load MLflow config
config_path = "./config/config.yaml"
model_config = mlflow.models.ModelConfig(development_config=config_path)
mlflow_config = model_config.get("mlflow_config") or {}

# Ensure we talk to the Databricks tracking server for run + model logging
mlflow.set_tracking_uri("databricks")

# Set experiment based on config
user_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
experiment_name = mlflow_config.get("experiment_name", "dspy_rag_agent_experiments")
experiment_info = mlflow.set_experiment(f"/Users/{user_name}/{experiment_name}")
print(f"MLflow Experiment info: {experiment_info}")

## Create evaluation dataset from existing traces, domain expert labels, build from scratch/use existing, or synthetic data
- Documentation here: https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/build-eval-dataset?language=Using+the+SDK

In [0]:
evaluation_uc_table_name = f"{catalog}.{schema}.{evaluation_table}"

# Load or create MLflow evaluation dataset
try:
    dataset = mlflow.genai.datasets.get_dataset(
        name=evaluation_uc_table_name
    )
except Exception as e:
    print(f"MLflow evaluation dataset {evaluation_uc_table_name} not found... Creating dataset")
    dataset = mlflow.genai.datasets.create_dataset(
        name=evaluation_uc_table_name,
        experiment_id=experiment_info.experiment_id
    )

### Important to note that human feedback will be included from Review app to MLflow Experiments tab if the endpoint is deployed correctly with MLflow 3 and experiment ID env variable is set

### Option 1: Add records to dataset from existing traces

In [0]:
from mlflow.deployments import get_deploy_client

# TO DO Update agent_endpoint_name
agent_endpoint_name = "agents_users-alex_miller-dspy_rag_agent"

client = get_deploy_client("databricks")
endpoint_info = client.get_endpoint(endpoint=agent_endpoint_name)

entity_info = endpoint_info['config']['served_entities']
model_fqn = entity_info[0]['entity_name']
model_version = entity_info[0]['entity_version']

print(f"UC Model Name: {model_fqn}")
print(f"UC Model Version: {model_version}")

In [0]:
import mlflow

client = mlflow.tracking.MlflowClient()
model_version_details = client.get_model_version(
    name=model_fqn, 
    version=model_version
)

model_id = getattr(model_version_details, "model_id", None)
# MLflow model version object will have unique .id and .run_id properties
print("Model ID:", model_id)
print("Run ID:", getattr(model_version_details, "run_id", None))

In [0]:
# 2. Search for traces
traces = mlflow.search_traces(
  # filter_string="attributes.status = 'OK'",
  model_id=model_id,
  order_by=["attributes.timestamp_ms DESC"],
  max_results=5
)
display(traces)

# 3. Add the traces to the evaluation dataset
dataset.merge_records(traces)
print(f"Added {len(traces)} records to evaluation dataset")

# Preview the dataset
df = dataset.to_df()
print(f"\nDataset preview:")
print(f"Total records: {len(df)}")
print("\nSample record:")
sample = df.iloc[0]
print(f"Inputs: {sample['inputs']}")

### Approach 2: Create from domain expert labels
- Documentation here: https://docs.databricks.com/aws/en/mlflow3/genai/human-feedback/expert-feedback/label-existing-traces

In [0]:

# from mlflow.genai.label_schemas import create_label_schema, InputCategorical, InputText

# # Collect feedback on the summary
# summary_quality = create_label_schema(
#     name="summary_quality",
#     type="feedback",
#     title="Is this summary concise and helpful?",
#     input=InputCategorical(options=["Yes", "No"]),
#     instruction="Please provide a rationale below.",
#     enable_comment=True,
#     overwrite=True,
# )

# # Collect a ground truth summary
# expected_summary = create_label_schema(
#     name="expected_summary",
#     type="expectation",
#     title="Please provide the correct summary for the user's request.",
#     input=InputText(),
#     overwrite=True,
# )

In [0]:
# from mlflow.genai.labeling import create_labeling_session

# # Create the Labeling Session with the schemas we created in the previous step
# label_summaries = create_labeling_session(
#     name="label_summaries",
#     assigned_users=[],
#     label_schemas=[summary_quality.name, expected_summary.name],
# )

In [0]:
# import mlflow

# # Use verison tracking to be able to easily query for the traces
# tracked_model = mlflow.set_active_model(name="my_app")

# # Run the app to generate traces
# sample_messages_1 = [
#     {"role": "user", "content": "what issues does john doe have?"},
# ]
# summary1_output = my_app(sample_messages_1)

# sample_messages_2 = [
#     {"role": "user", "content": "what issues does acme corp have?"},
# ]
# summary2_output = my_app(sample_messages_2)

# # Query for the traces we just generated
# traces = mlflow.search_traces(model_id=tracked_model.model_id)

# # Add the traces to the session
# label_summaries.add_traces(traces)

# # Print the URL to share with your domain experts
# print(f"Share this Review App with your team: {label_summaries.url}")

In [0]:
# labeled_traces_df = mlflow.search_traces(run_id=label_summaries.mlflow_run_id)
# dataset.merge_records(labeled_traces_df)

### Approach 3: Build from scratch or import existing

In [0]:
# # Define comprehensive test cases
# evaluation_examples = [
#     {
#         "inputs": {"question": "What is MLflow?"},
#         "expected": {
#             "expected_response": "MLflow is an open source platform for managing the end-to-end machine learning lifecycle.",
#             "expected_facts": [
#                 "open source platform",
#                 "manages ML lifecycle",
#                 "experiment tracking",
#                 "model deployment"
#             ]
#         },
#     },
# ]

# eval_dataset.merge_records(evaluation_examples)

In [0]:
import json

existing_inference_table_with_traces = "users.alex_miller.dspy_rag_agent_optimized_payload"

# TO DO: Update the filter logic to filter on correct requests
inference_table = spark.table(existing_inference_table_with_traces).select("request", "response").toPandas()

def merge_json_columns(row):
    try:
        # Parse JSON strings to dictionaries
        request_dict = json.loads(row["request"]) if row["request"] else {}
        response_dict = json.loads(row["response"]) if row["response"] else {}
        
        # Merge the dictionaries
        return {**request_dict, **response_dict}
    except (json.JSONDecodeError, TypeError):
        # Handle cases where JSON parsing fails
        return {}

inference_table["inputs"] = inference_table.apply(merge_json_columns, axis=1)
inference_table = inference_table.drop(columns=['request', 'response'])
display(inference_table)

# merge records to dataset
dataset.merge_records(inference_table)